In [1]:
import jax.numpy as jnp

In [2]:
from model import Stem, BarebonesFlowModel
from train import Training


In [3]:
import treescope
from treescope import IPythonVisualization

treescope.active_autovisualizer.set_globally(treescope.ArrayAutovisualizer())
treescope.basic_interactive_setup(autovisualize_arrays=True)



In [4]:
training = Training.default_build()
model = training.model


Logging to runs/v1_20251114_150800


In [22]:
training.train_one_epoch(0, 0)

/usr/lib/python3.13/multiprocessing/popen_fork.py:67: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
/usr/lib/python3.13/multiprocessing/popen_fork.py:67: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


Epoch [1/200] | Avg Loss: 0.822074


79

In [5]:
def sample_to_jax(s):
    return jnp.array(s[0].numpy()), jnp.array(s[1].numpy()), jnp.array(s[2].numpy())


def display_sample(f1, f2, flow):
    return treescope.figures.inline(
        treescope.render_array(f1, rows=(0,), columns=(1, 2)),
        treescope.render_array(f2, rows=(0,), columns=(1, 2)),
        treescope.render_array(flow.reshape(18,18, 2), rows=(0,), columns=(1, 2)),
        wrap=True,
    )


In [6]:
dataset = training.train_dataset
sample = dataset[6]
f1, f2, flow = sample_to_jax(sample)

display_sample(f1, f2, flow)


<Arrayviz rendering><Arrayviz rendering><Arrayviz rendering>

In [10]:
def find_displayable_chunk(size, start=8, end=2):
    for c in range(start, end, -1):
        if size % c == 0:
            return size // c, c
    return size
    
def display_filters(stem):
    H, W, I, O = stem.dw1.kernel.value.shape
    O1, O2 = find_displayable_chunk(O)
    chunked_dw1 = stem.dw1.kernel.value.reshape(H,W,O1,O2)    
    
    return treescope.figures.inline(
        treescope.render_array(chunked_dw1, rows=(0,), columns=(1,2), sliders=(3,)) 
    )
    pass

display_filters(model.stem)


<Arrayviz rendering>

In [11]:

def display_stem(s1, s2):
    s =jnp.stack([s1, s2], axis=0)
    B, H, W, C = s.shape
    chunk, size = find_displayable_chunk(C, start=6)
    chunked = s.reshape(B, H, W, chunk, size)
    return treescope.figures.inline(
        treescope.render_array(chunked, rows=(1, 0), columns=(2, 3), sliders=(4,)),
    )


display_stem(model.stem(f1), model.stem(f2))

model.stem(f1).shape

(8, 8, 32)

In [21]:
import jax
p1 = model.stem(f1)
p2 = model.stem(f1)
H, W, C = p1.shape

p1 = p1.reshape(H*W, C)
p2 = p2.reshape(H*W, C)

s = p1 @ p2.transpose(1,0)
zero_bias = model._zero_flow_bias()
biased_sim = s * zero_bias
attn = jax.nn.softmax(biased_sim * model.attn_temperature, axis=-1)
attn_patches = attn.reshape(8,8,8,8)


treescope.figures.inline(
    # treescope.render_array(p1, rows=(0,), columns=(1,)),
    # treescope.render_array(p2.transpose(1,0), rows=(0,), columns=(1,)),            
    # treescope.render_array(s.reshape(H,W,H,W), rows=(0,1), columns=(2,3)),        
    # treescope.render_array(biased_sim.reshape(H,W,H,W), rows=(2,0), columns=(3,1)),     
    treescope.render_array(attn.reshape(H,W,H,W), rows=(2,0), columns=(3,1))
    # treescope.render_array(sm.reshape(H,W,H,W), rows=(2, 0), columns=(3, 1)),     
)



<Arrayviz rendering>

In [16]:
L_broadcast = jnp.broadcast_to(
    model.location_grid.value.reshape(1, 64, 2),
    (1, 64, 2)
)

# (B, 64, 2) - Flow in "Grid Units"
flow_pred_flat = (attn @ L_broadcast) - L_broadcast
treescope.render_array(flow_pred_flat.reshape(8,8,2), rows=(0,), columns=(1,2))

# treescope.render_array(L_broadcast.reshape(8,8,2), rows=(0,), columns=(1,2))


<Arrayviz rendering>

In [20]:
model.attn_temperature = 0.1
model.zero_boost_radius.value = 200
zero = model._zero_flow_bias()
zero = zero.reshape(8,8,8,8)
treescope.render_array(zero, rows=(2, 0), columns=(3, 1))

<Arrayviz rendering>

In [25]:
h,w = model.grid_size_hw
location_grid = jnp.broadcast_to(
     model.location_grid.value.reshape(1, w*h, 2),
     (1, w*h, 2)
)


In [22]:
flow, aux = model(f1[None,...], f2[None,...], flow[None,...])

In [23]:
treescope.render_array(flow, rows=(1,), columns=(2,3))

<Arrayviz rendering>

In [28]:
attn = aux['trace']['attn']
attn = attn.reshape(8,8,8,8)
treescope.render_array(attn, rows=(2,0), columns=(3,1))

<Arrayviz rendering>

In [29]:
aux

{'loss': {'flow': Array(0., dtype=float32),
  'variance': Array(0.01466938, dtype=float32),
  'covariance': Array(0.12199476, dtype=float32)},
 'trace': {'attn': Array([[[9.99999166e-01, 1.73905927e-14, 5.36492968e-11, ...,
           4.62778557e-12, 2.57761694e-13, 3.97393215e-15],
          [5.49693002e-09, 9.99936342e-01, 5.69999083e-11, ...,
           3.45063977e-10, 5.69063789e-11, 5.49686408e-10],
          [6.45940364e-20, 1.35507137e-26, 1.00000000e+00, ...,
           1.62211243e-17, 1.01440817e-22, 3.33929571e-17],
          ...,
          [3.32120526e-10, 5.57822661e-14, 1.52288294e-07, ...,
           9.99952555e-01, 2.52573407e-10, 4.28890489e-05],
          [6.74327623e-07, 5.98555705e-10, 2.79187361e-07, ...,
           4.08776395e-06, 9.93111849e-01, 2.60235898e-13],
          [9.09998482e-34, 7.43342725e-34, 9.98629651e-27, ...,
           3.69260031e-24, 0.00000000e+00, 1.00000000e+00]]], dtype=float32),
  'f1_patches': Array([[[-0.6916036 , -0.23665595,  0.34608185, ...,  0.8986829 ,
            1.7568028 ,  0.8974994 ],
          [-0.36430842,  0.1913117 , -0.85388476, ...,  1.0093125 ,
           -0.03469267,  0.17013988],
          [-0.09455106, -1.5795642 ,  1.9367561 , ...,  0.01765751,
            1.1894513 ,  1.587139  ],
          ...,
          [-0.653447  , -1.2686713 ,  1.1902987 , ..., -0.25643572,
            1.1949357 ,  0.9298742 ],
          [-0.5607298 , -0.9771715 ,  0.19931073, ..., -1.0811194 ,
            0.12857372,  0.14379366],
          [ 1.1454825 ,  0.06976992,  1.1063131 , ..., -0.34988257,
            1.1756176 ,  2.341783  ]]], dtype=float32),
  'f2_patches': Array([[[-0.7887664 , -0.21328318,  0.15138626, ...,  1.0523846 ,
            1.7117703 ,  0.8255184 ],
          [-0.4849409 ,  0.2745353 , -1.057018  , ...,  1.2452732 ,
            0.00521846,  0.11298823],
          [-0.16680658, -1.2338338 ,  1.9346726 , ...,  0.08896006,
            1.1937008 ,  1.3584181 ],
          ...,
          [-0.65665877, -0.94186085,  1.5617443 , ..., -0.18211767,
            1.1703919 ,  1.0113251 ],
          [-0.56590635, -0.84193534,  0.12534523, ..., -0.90271354,
            0.28480592,  0.20042051],
          [ 1.0347494 ,  0.18134324,  1.1746275 , ..., -0.40624526,
            1.2114332 ,  2.1629384 ]]], dtype=float32)}}